In [ ]:
#| default_exp hub

# hub

> Find a model, fetch it, and work out what its author expected you to do to a picture first.

`Model('org/some-classifier')` with no suffix to go on ends up here: `resolve_model` lists what the
repo ships, picks a weights file, downloads it with its sidecars, and hands back a runtime and a
path. `find_models` is the step before that, for when the model is not known by name yet.

In [ ]:
#| export
from __future__ import annotations
import json, os, re
from functools import lru_cache
from pathlib import Path

from fastcore.all import AttrDict, L

from anya.core import infer_runtime, runtimes

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp
from anya.core import sidecar_labels
from anya.vision import read_labels

## Picking a file out of a repo

A repo often ships the same model several ways: float and quantised, one input size and another.
`pick_file` scores the candidates rather than taking the first match, and `prefer` moves the winner
without anyone having to name a filename.

In [ ]:
#| export
WEIGHT_EXTS = {'.onnx': 'onnx', '.ort': 'onnx', '.tflite': 'litert', '.lite': 'litert',
               '.mlpackage': 'coreml', '.mlmodel': 'coreml', '.mlmodelc': 'coreml'}

def weight_files(files) -> L:
    'Every file in `files` that some runtime can open.'
    return L(files).filter(lambda f: any(str(f).lower().endswith(e) for e in WEIGHT_EXTS))

def file_runtime(f) -> str|None:
    'Which runtime opens `f`, by suffix.'
    s = str(f).lower()
    return next((r for e, r in WEIGHT_EXTS.items() if s.endswith(e)), None)

def dflt_prefer() -> tuple:
    'Runtime preference for this machine: Core ML on a Mac, then ONNX, then LiteRT.'
    import sys
    return ('coreml', 'onnx', 'litert') if sys.platform == 'darwin' else ('onnx', 'litert', 'coreml')

def score_file(f, prefer=None) -> tuple:
    'Sort key for a candidate weights file: runtime preference first, then how plain the name is.'
    prefer = prefer or dflt_prefer()
    rt = file_runtime(f)
    s, name = str(f).lower(), Path(str(f)).stem.lower()
    rank = prefer.index(rt) if rt in prefer else len(prefer)
    plain = 0 if name in ('model', 'model_float32', Path(str(f)).parent.name.lower()) else 1
    deep = s.count('/')                                   # a file at the root beats one buried in a variant folder
    return (rank, plain, deep, len(s))

def pick_file(files,            # every path in the repo
              prefer=None,      # runtime order, e.g. ('litert','onnx')
              pat:str=None      # only consider files matching this regex
             ) -> str|None:
    'The one weights file to download, or None when the repo ships none.'
    c = weight_files(files)
    if pat: c = c.filter(lambda f: re.search(pat, str(f), re.I))
    if isinstance(prefer, str): prefer = (prefer,) + tuple(r for r in dflt_prefer() if r != prefer)
    return min(c, key=lambda f: score_file(f, prefer), default=None)

In [ ]:
#| hide
_fs = ['README.md', 'config.json', 'onnx/model.onnx', 'onnx/model_quantized.onnx', 'model.tflite']
test_eq(weight_files(_fs), ['onnx/model.onnx', 'onnx/model_quantized.onnx', 'model.tflite'])
test_eq(file_runtime('a/b.TFLITE'), 'litert')
test_eq(pick_file(_fs, prefer='litert'), 'model.tflite')
test_eq(pick_file(_fs, prefer='onnx'), 'onnx/model.onnx')     # the plain name beats the quantised one
test_eq(pick_file(_fs, prefer='onnx', pat='quant'), 'onnx/model_quantized.onnx')
test_eq(pick_file(['README.md']), None)

## What the author expected

`preprocessor_config.json` and `config.json` are how a HuggingFace repo says what size it wants,
how it was normalised, and what its classes are called. Reading them is the difference between a
model that works out of the box and one that returns confident nonsense.

In [ ]:
#| export
def _hw(sz) -> tuple|None:
    'A `size` field as `(h, w)`: it is an int, a height/width pair, or a shortest edge.'
    if isinstance(sz, int): return (sz, sz)
    if not isinstance(sz, dict): return None
    h, w = sz.get('height'), sz.get('width')
    if h and w: return (int(h), int(w))
    return (int(s), int(s)) if (s := sz.get('shortest_edge')) else None

def prep_kwargs(cfg:dict) -> dict:
    'Turn a `preprocessor_config.json` into `size=` / `norm=` / `resize=` / `crop_pct=` for a `Model`.'
    if not cfg: return {}
    out, size, crop = {}, _hw(cfg.get('size')), _hw(cfg.get('crop_size'))
    if cfg.get('do_center_crop'): out['resize'] = 'center_crop'
    # size 256 with crop_size 224 means resize the short side to 256, then crop 224 out of the middle.
    # SpotLab/YOLOv8Detection asks for a 640 crop out of a 256 short side, which is not a crop.
    if crop and size and crop[0] < size[0]: out['crop_pct'] = round(crop[0]/size[0], 4)
    if (sz := (crop if cfg.get('do_center_crop') else None) or size or crop): out['size'] = sz
    m, s = cfg.get('image_mean'), cfg.get('image_std')
    if m and s: out['norm'] = (tuple(float(x) for x in m), tuple(float(x) for x in s))
    # bicubic against bilinear moved top-1 on one of five photos, so the filter is worth carrying
    if isinstance(cfg.get('resample'), int): out['resample'] = cfg['resample']
    return out

In [ ]:
#| hide
test_eq(prep_kwargs({'size': {'height': 260, 'width': 260}})['size'], (260, 260))
test_eq(prep_kwargs({'size': {'shortest_edge': 224}, 'do_center_crop': True})['resize'], 'center_crop')
test_eq(prep_kwargs({'image_mean': [0.5]*3, 'image_std': [0.5]*3})['norm'], ((0.5,)*3, (0.5,)*3))
test_eq(prep_kwargs({}), {})
# timm's, as optimum writes it: the crop is what the model sees, the short side is resized past it
_pk = prep_kwargs({'size': {'shortest_edge': 256}, 'crop_size': {'height': 224, 'width': 224},
                   'do_center_crop': True, 'resample': 3})
test_eq(_pk['size'], (224, 224)); test_eq(_pk['crop_pct'], 0.875); test_eq(_pk['resize'], 'center_crop')
test_eq(_pk['resample'], 3)
# SpotLab/YOLOv8Detection asks for a 640 crop out of a 256 short side, so there is no crop fraction
test_eq('crop_pct' in prep_kwargs({'size': {'shortest_edge': 256}, 'crop_size': {'height': 640, 'width': 640},
                                   'do_center_crop': True}), False)

## Fetching

`fetch` downloads one file from the Hub and caches it. `resolve_model` is what `Model` calls: it
picks the file, brings its sidecars along, and returns `(runtime, path)`. It is cached, so the two
places that ask during one construction cause one download.

In [ ]:
#| export
SIDECARS = ('config.json', 'preprocessor_config.json', 'labels.txt', 'classes.txt', 'labelmap.txt')

def _api(token=None):
    try: from huggingface_hub import HfApi
    except ImportError as e:
        raise ImportError("anya needs huggingface-hub to fetch a model by repo id: pip install 'anya[hub]'") from e
    return HfApi(token=token)

def repo_files(repo_id:str, revision:str=None, token=None) -> L:
    'Every file in a Hub repo.'
    return L(_api(token).list_repo_files(repo_id, revision=revision))

def fetch(repo_id:str,          # a Hub repo id
          filename:str,         # a path inside the repo
          revision:str=None,
          token=None
         ) -> Path:
    'Download one file from the Hub (cached) and return the local path.'
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(repo_id, filename, revision=revision, token=token))

def fetch_sidecars(repo_id:str, files, dest:Path, revision:str=None, token=None) -> dict:
    'Download the config and label files that sit beside the weights, ignoring any that 404.'
    out = {}
    for n in SIDECARS:
        hit = next((f for f in files if str(f).rsplit('/', 1)[-1] == n), None)
        if not hit: continue
        try: out[n] = fetch(repo_id, hit, revision=revision, token=token)
        except Exception: pass          # a listed file can still be gated or moved
    return out

In [ ]:
#| export
@lru_cache(maxsize=64)
def resolve_model(repo_id:str,        # a Hub repo id, or a local path
                  file:str=None,      # a filename inside the repo, when you know it
                  revision:str=None,
                  prefer=None,        # runtime order, e.g. 'litert'
                  token=None
                 ) -> tuple:
    'Work out `(runtime, local path)` for a Hub repo, downloading the weights and their sidecars.'
    p = Path(repo_id).expanduser()
    if p.exists(): return (infer_runtime(p), str(p))
    files = repo_files(repo_id, revision=revision, token=token)
    pick = file or pick_file(files, prefer=prefer)
    if pick is None: raise ValueError(
        f'{repo_id} ships no file anya can run (looked for {", ".join(WEIGHT_EXTS)}). '
        f'It has: {", ".join(map(str, files[:12]))}')
    path = fetch(repo_id, pick, revision=revision, token=token)
    fetch_sidecars(repo_id, files, path.parent, revision=revision, token=token)
    return (file_runtime(pick), str(path))

def model_config(path) -> AttrDict:
    'The `config.json` and `preprocessor_config.json` cached next to a downloaded model.'
    d, out = Path(path).parent, {}
    for _ in range(3):                              # the hub cache nests the file under snapshots/<sha>/
        for n in ('config.json', 'preprocessor_config.json'):
            if (d/n).exists() and n not in out:
                try: out[n] = json.loads((d/n).read_text())
                except Exception: pass
        d = d.parent
    return AttrDict(config=out.get('config.json', {}), preprocessor=out.get('preprocessor_config.json', {}))

In [ ]:
#| hide
_d = Path(mkdtemp())
(_d/'m.onnx').write_bytes(b'x')
(_d/'config.json').write_text('{"id2label": {"0": "wren"}}')
(_d/'preprocessor_config.json').write_text('{"size": {"height": 260, "width": 260}}')
test_eq(resolve_model(str(_d/'m.onnx')), ('onnx', str(_d/'m.onnx')))       # a local path never touches the network
_c = model_config(_d/'m.onnx')
test_eq(read_labels(_c.config), ['wren'])
test_eq(prep_kwargs(_c.preprocessor)['size'], (260, 260))
test_eq(sidecar_labels(_d/'m.onnx'), _d/'config.json')

## Finding one

`find_models` asks the Hub for models that both match the words and ship a file anya can open. With
no Hub reachable, or nothing found, `web_models` falls back to fossick and reads repo ids out of the
search results, which is the same route a person takes.

In [ ]:
#| export
PIPELINES = {'classify': 'image-classification', 'detect': 'object-detection',
             'segment': 'image-segmentation', 'embed': 'image-feature-extraction'}

def find_models(query:str,          # what the model should do, in words
                task:str=None,      # 'classify', 'detect', 'segment', 'embed'
                runtime:str=None,   # only repos with a file this runtime can open
                n:int=10,
                token=None
               ) -> L:
    'Search the Hub for models anya can run, most downloaded first.'
    kw = dict(search=query, sort='downloads', limit=max(n*3, 30))
    if task: kw['pipeline_tag'] = PIPELINES.get(task, task)
    if runtime: kw['filter'] = {'onnx': 'onnx', 'litert': 'tflite', 'coreml': 'coreml'}[runtime]
    ms = L(_api(token).list_models(**kw))
    return ms.map(lambda m: AttrDict(id=m.id, downloads=getattr(m, 'downloads', 0),
                                     task=getattr(m, 'pipeline_tag', None), likes=getattr(m, 'likes', 0)))[:n]

def web_models(query:str, n:int=5) -> L:
    'Ask the open web for HuggingFace models, for when the Hub API is not reachable.'
    try: from fossick import search
    except ImportError as e:
        raise ImportError('web_models needs fossick: pip install fossick') from e
    hits = search(f'{query} huggingface model onnx OR tflite', n=n*3)
    ids = L(re.findall(r'huggingface\.co/([\w.\-]+/[\w.\-]+)', ' '.join(h.get('href', '') for h in hits)))
    return ids.unique()[:n].map(lambda i: AttrDict(id=i, downloads=None, task=None, likes=None))

In [ ]:
#| eval: false
find_models('bird classifier', task='classify', runtime='litert', n=5)

## Naming one

A repo id is a poor thing to keep retyping, and a poorer thing to put in a prompt. `alias` writes a
short name into `~/.anya/models.json`; `Model` accepts it anywhere a repo id goes.

In [ ]:
#| export
def registry_path() -> Path:
    'Where aliases live: `$ANYA_HOME/models.json`, else `~/.anya/models.json`.'
    return Path(os.environ.get('ANYA_HOME', Path.home()/'.anya')).expanduser()/'models.json'

def aliases() -> dict:
    'Every alias defined on this machine.'
    p = registry_path()
    if not p.exists(): return {}
    try: return json.loads(p.read_text())
    except Exception: return {}

def alias(name:str,            # the short name to use from now on
          repo:str=None,       # the repo id or path it stands for; None deletes it
          **kw                 # file=, labels=, norm=, size=, task=, anything Model takes
         ) -> dict:
    'Define (or with `repo=None`, delete) a short name for a model, saved for later sessions.'
    d, p = aliases(), registry_path()
    if repo is None: d.pop(name, None)
    else: d[name] = dict(model=repo, **kw)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(d, indent=1))
    return d

def resolve_alias(name) -> dict|None:
    'The `Model` keyword arguments an alias stands for, or None.'
    return aliases().get(str(name))

In [ ]:
#| hide
os.environ['ANYA_HOME'] = str(_d)
alias('aussie-birds', 'org/birds-v2', file='model.tflite', labels='labels.txt')
test_eq(resolve_alias('aussie-birds')['model'], 'org/birds-v2')
test_eq(resolve_alias('aussie-birds')['file'], 'model.tflite')
test_eq(resolve_alias('nope'), None)
alias('aussie-birds', None)
test_eq(aliases(), {})

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()